# video-app lab 09 - quality upgrade
Baseline we measured: 512x512, 16 frames, 8 fps, 4 steps = 2.0 s, 304 KB, 1 fps of motion detail.
Three levers, all free: (1) 8-step distilled weights instead of 4, (2) 32 frames instead of 16
(4 s of motion), (3) 640px + frame interpolation to 24 fps + light sharpen in ffmpeg.
Trade: ~4-6 min on T4 instead of ~40 s. Worth it for a real look.


In [ ]:
import os, shutil, psutil, torch
print(f'RAM {psutil.virtual_memory().used/1024**3:.1f}/{psutil.virtual_memory().total/1024**3:.1f} GB | disk {shutil.disk_usage("/").free/1024**3:.0f} GB | cuda {torch.cuda.is_available()}')
if not torch.cuda.is_available(): raise SystemExit('STOP: no GPU. Runtime > Change runtime type > T4 GPU + High-RAM > Save.')
print('GPU', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1024**3,1), 'GB')


In [ ]:
import gc, time, torch
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
DTYPE = torch.float16
def load(cls, repo, **kw):
    kw = dict(kw, low_cpu_mem_usage=True)
    try: return cls.from_pretrained(repo, dtype=DTYPE, **kw)
    except TypeError: return cls.from_pretrained(repo, torch_dtype=DTYPE, **kw)
adapter = load(MotionAdapter, 'guoyww/animatediff-motion-adapter-v1-5-2')
pipe = load(AnimateDiffPipeline, 'emilianJR/epiCRealism', motion_adapter=adapter, use_safetensors=True)
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing='trailing', beta_schedule='linear')
try: pipe.vae.enable_slicing()
except Exception: pass
pipe.to('cuda'); gc.collect()
print('loaded | VRAM', round(torch.cuda.memory_allocated()/1024**3,2), 'GB')


In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
STEPS, GUID = 25, 7.5
try:
    sd = load_file(hf_hub_download('ByteDance/AnimateDiff-Lightning','animatediff_lightning_8step_diffusers.safetensors'))
    sd = {('unet.'+k): v.to(DTYPE) for k, v in sd.items()}
    res = pipe.unet.load_state_dict(sd, strict=False)
    print('missing', len(res.missing_keys), 'unexpected', len(res.unexpected_keys))
    if len(res.unexpected_keys) == 0: STEPS, GUID = 8, 1.0; print('8-STEP LIGHTNING OK')
except Exception as e:
    print('8-step skipped:', type(e).__name__, str(e)[:120])
pipe.to('cuda')


In [ ]:
from diffusers.utils import export_to_video
PROMPT = ('cinematic wildlife photo of a red fox trotting through deep snow between snow covered pines, '
          'low winter sun rays through branches, shallow depth of field, fur detail, 35mm film grain, '
          'steady tracking camera, natural colour')
NEG = 'blurry, low resolution, jpeg artifacts, distorted anatomy, extra legs, watermark, text, flicker, washed out'
t0 = time.time()
frames = pipe(prompt=PROMPT, negative_prompt=NEG, num_frames=32, height=640, width=640,
              guidance_scale=GUID, num_inference_steps=STEPS,
              generator=torch.Generator('cpu').manual_seed(7)).frames[0]
print('GENERATED', STEPS, 'steps in', round(time.time()-t0), 's | peak VRAM', round(torch.cuda.max_memory_allocated()/1024**3,2), 'GB')
export_to_video(frames, '/content/raw_09.mp4', fps=8)
print('raw MB', round(os.path.getsize('/content/raw_09.mp4')/1024**2,1))
import subprocess
subprocess.run(['ffmpeg','-y','-v','error','-i','/content/raw_09.mp4','-vf',
                'minterpolate=fps=24:mi_mode=mci:mc_mode=aobmc:vsbmc=1,unsharp=5:5:0.8',
                '-c:v','libx264','-crf','16','-pix_fmt','yuv420p','/content/clip_09_fox_24fps.mp4'], check=True)
print('smooth MB', round(os.path.getsize('/content/clip_09_fox_24fps.mp4')/1024**2,1))
from IPython.display import Video, display
display(Video('/content/clip_09_fox_24fps.mp4', embed=True, width=640))
from google.colab import files; files.download('/content/clip_09_fox_24fps.mp4')


### Next lever if still not good
Same free GPU (L4/A100 available on your PRO account) running **LTX-Video** or **Wan2.2 5B** - native 720p,
much better motion. That is notebook 10, and it is the step-change, not a tweak.
